# Olist E-Commerce - 07 : Conclusions & Executive Summary

**Author:** Diego Ospina | **Dataset:** Brazilian E-Commerce Public Dataset by Olist

> This final notebook condenses the whole project into an **executive summary**: the KPIs, the key insights from each stage (EDA, statistics, BI, ML), and actionable recommendations. Figures are recomputed inline (from the cleaned datasets) so every number below is reproducible.

## 1. Snapshot of the business

In [1]:
import pandas as pd
import os

PROC = os.path.join('..', 'data', 'processed')
m = pd.read_parquet(os.path.join(PROC, 'olist_master_orders.parquet'))
it = pd.read_parquet(os.path.join(PROC, 'olist_items.parquet'))
d = m[m['order_status'] == 'delivered']

kpi = {
    'Total orders': f"{len(m):,}",
    'Delivered orders': f"{len(d):,}",
    'Unique customers': f"{m['customer_unique_id'].nunique():,}",
    'Sellers': f"{it['seller_id'].nunique():,}",
    'Products in catalog': f"{it['product_id'].nunique():,}",
    'Total revenue (BRL)': f"{d['order_value'].sum():,.0f}",
    'Average order value (BRL)': f"{d['order_value'].mean():,.2f}",
    'Avg items per order': f"{d['n_items'].mean():.2f}",
    'Late deliveries (%)': f"{(d['delivery_status']=='Late').mean()*100:.2f}",
    'Avg review score': f"{m.dropna(subset=['review_score_mean'])['review_score_mean'].mean():.2f}",
    'States covered': f"{m['customer_state'].nunique()}",
}
pd.DataFrame(kpi.items(), columns=['KPI', 'Value']).set_index('KPI')

,Value
KPI,
Total orders,"99,441"
Delivered orders,"96,478"
Unique customers,"96,096"
Sellers,"3,095"
Products in catalog,"32,951"
Total revenue (BRL),"15,419,774"
Average order value (BRL),159.83
Avg items per order,1.14
Late deliveries (%),8.11


## 2. Key insights by analysis stage

### Understanding (01) and Cleaning (02)
- The data spans **Sep 2016 - Oct 2018**; ~97% of orders are `delivered`.
- Zip codes are strings; category names were translated to English and missing ones labelled `not_specified`; geolocation was de-duplicated to one point per zip.

### EDA (03)
- Sales grow strongly over time (best month ~988k BRL).
- Order values are heavily right-skewed (median ~105 BRL vs mean ~160 BRL).
- ~92% of delivered orders arrive on time (8.11% late).
- Reviews are positive overall (mean ~4.1) with a notable 1-star cluster.
- São Paulo animates the marketplace; credit card is the dominant payment method.

### Statistics (04)
- `order_value` is **non-normal**, so non-parametric tests were used.
- Price and delivery delay are practically uncorrelated.
- **Late deliveries earn significantly lower review scores** (Mann-Whitney U, p<0.001).
- Payment method and punctuality are statistically (but not practically) associated.

### Business intelligence (05)
- RFM finds a **Champion/VIP** group that drives a disproportionate share of revenue plus a large **Lost/At-risk** tail (most customers buy once).
- Cohort retention is front-loaded: many customers purchase once and never return.
- Average CLV per cohort is stable; repeat purchase is the main lever to raise it.

### Predictive modeling (06)
- **Classification** (late order): Random Forest reaches ROC-AUC ~0.74-0.75; composition features (`n_items`, `freight_value`, `weight_total_g`, `customer_state`) matter most.
- **Regression** (order value): Random Forest explains a moderate share of variance (R2 ~0.37) and beats a median baseline, reflecting the difficulty of forecasting basket size.


## 3. Sanity checks on the cleaned data

In [2]:
# Verificacion final: integridad y rangos del dataset limpio
assert len(m) == 99441, 'master order count'
assert m['order_value'].notna().sum() > 0
assert m['delivery_status'].eq('On time').any() and m['delivery_status'].eq('Late').any()
print('Range of purchase dates:', m['order_purchase_timestamp'].min().date(), 'a', m['order_purchase_timestamp'].max().date())
print('Master rows:', len(m), '| cols:', m.shape[1])
print('Todos los chequeos pasaron OK.')

Range of purchase dates: 2016-09-04 a 2018-10-17
Master rows: 99441 | cols: 33
Todos los chequeos pasaron OK.


## 4. Executive recommendations

1. **Logistics (impactful & cheap):** the ~8% late orders are strongly associated with lower review scores. Prioritizing on-time delivery for the largest/weight-heaviest orders should lift average satisfaction.
2. **Customer retention:** most customers buy once. A reactivation program aimed at the `Lost` / `At-risk` RFM segments and post-purchase incentives could raise repeat rates and average CLV more than acquiring new customers.
3. **Category strategy:** concentrate merchandising on the top revenue categories (led by bed_bath_table and health_beauty) while using the long tail for assortment depth.
4. **Forecasting:** the regression model gives a useful (if imperfect) baseline for expected basket size; combining it with logistics features would support stock and fleet planning.


## 5. What is included in this portfolio

| Notebook | Deliverable |
|---|---|
| `01_data_understanding` | Raw data inventory, granularity, data quality, referential integrity |
| `02_cleaning_preprocessing` | Reproducible pipeline -> 5 tidy datasets in `data/processed/` |
| `03_eda_visualizations` | Full EDA with 8+ figures saved in `images/` |
| `04_statistical_analysis` | scipy hypothesis tests & inference |
| `05_business_intelligence` | KPIs, RFM, cohort retention, CLV |
| `06_predictive_modeling` | scikit-learn classification + regression pipelines |
| `07_conclusions` | Executive summary & recommendations |
| `sql_python_demo` | SQL + pandas interoperability (SQL scripts in `sql/`) |

**Stack:** Python, pandas, NumPy, matplotlib, seaborn, SciPy, scikit-learn, SQL.

## 6. How to reproduce

```bash
python -m venv .venv && .venv\Scripts\activate
pip install -r requirements.txt
python src/build_processed.py      # -> data/processed/*.parquet
# then open the notebooks in order (01 to 07, plus sql_python_demo)
```